# Random search de hiperparâmetros — datasets 13cat

Este notebook realiza a busca aleatória de hiperparâmetros para Ridge, Random Forest, Extra Trees e Gradient Boosting em cada uma das sete áreas térmicas, sobre a versão 13cat da base.

A formulação é a da solução entregue na Sprint 3: um modelo por área, treinado no formato longo (uma linha por par ensaio × componente), com a identidade do componente substituída pelo patamar (média) e pela escala (desvio-padrão) calculados apenas sobre o treino de cada dobra. A validação é leave-one-group-out sobre as 17 configurações de treino; os ensaios de teste não são lidos. Valores ausentes do alvo são descartados (máscara), nunca imputados.

Os arquivos são lidos da pasta do próprio caderno, notebooks/colabs_S4: os sete dataset_<area>_13cat.csv, o particao_treino_teste.csv e o limites_ativos.csv, todos gerados pelo tratamento_datasets_13cat. Os resultados intermediários são gravados na subpasta resultados_random_search, que não é versionada; com RETOMAR igual a verdadeiro, uma execução interrompida continua do ponto em que parou, e a subpasta deve ser apagada sempre que a base ou o espaço de busca mudarem.


## 1. Configuração

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import json, os, time, warnings
import numpy as np, pandas as pd
from scipy.stats import loguniform, randint, uniform
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut, RandomizedSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

PASTA_DADOS = "."          # os CSVs sao lidos apenas do colabs_S4, a pasta deste caderno
PASTA_SAIDA = "resultados_random_search"
VERSAO = "13cat"
SEMENTE = 42
N_JOBS = -1                 # paralelismo nas dobras; os estimadores usam 1 núcleo cada
TOP_K = 3                   # candidatos do topo reavaliados com os KPIs completos
RETOMAR = True              # pula pares já gravados; apague PASTA_SAIDA ao mudar N_ITER ou espaços
MARGEM_KPI = 10.0           # margem contratual, em %
MARGENS_SEGURANCA = (0.0, 4.0, 8.0)   # °C somados à previsão antes da comparação com o limite

AREAS = ["oven_hmi", "ct_knobs", "ipc_cooling", "main_vent_sys",
         "mainback", "ipc_comp", "coil_temps"]
ALGORITMOS = ["ridge", "random_forest", "extra_trees", "gradient_boosting"]
N_ITER = {"ridge": 20, "random_forest": 40, "extra_trees": 40, "gradient_boosting": 40}

# O motivo do ensaio mudou de nome na quarta sprint: cfg_ era o nome dos seis
# indicadores sobrepostos, reason_ é o nome dos seis níveis exclusivos. Os dois
# ficam declarados para que o caderno leia tanto a base antiga quanto a atual.
PREFIXOS_FEATURES = ("blower_direction_", "housing_", "board_position_", "run_case_",
                     "losses_", "deflector_hole_", "material_coil_support_",
                     "cfg_", "reason_")

# Colunas deliberadamente fora das entradas. A ordem de acionamento das bobinas foi
# medida como prejudicial na modelagem da quarta sprint. Qualquer outra coluna que não
# seja entrada nem alvo interrompe a execução, para que uma mudança de nome nunca
# volte a descartar variáveis em silêncio.
EXCLUIDAS = ("loss_profile_order_",)
TOTAL_ESPERADO = 3277       # leituras de treino do protocolo da Seção 4.3

assert os.path.isdir(PASTA_DADOS), f"Pasta '{PASTA_DADOS}' não encontrada em {os.path.abspath('.')}"
os.makedirs(PASTA_SAIDA, exist_ok=True)
print("Diretório:", os.path.abspath("."))

## 2. Partição, limites e formato longo

In [ ]:
def caminho(arquivo):
    c = os.path.join(PASTA_DADOS, arquivo)
    if not os.path.exists(c):
        raise FileNotFoundError(f"Arquivo ausente: {c}")
    return c


particao = pd.read_csv(caminho("particao_treino_teste.csv"))
IDX_TREINO = particao.loc[particao["conjunto"] == "treino", "run_idx"].astype(int).to_numpy()
DOBRA = particao.set_index("run_idx")["dobra"]

_lim = pd.read_csv(caminho("limites_ativos.csv"))
LIMITE_DE = dict(zip(_lim["tc_name"], _lim["limite_num"]))
SO_MONITORAR = set(_lim.loc[_lim["apenas_monitorar"] == True, "tc_name"])

print(f"Treino: {len(IDX_TREINO)} ensaios | teste reservado: {(particao['conjunto'] == 'teste').sum()} "
      f"| dobras: {DOBRA.dropna().nunique()}")


def carrega_longo(area):
    """Devolve X (configuração + componente), y, grupos e o formato longo, só com ensaios de treino."""
    df = pd.read_csv(caminho(f"dataset_{area}_{VERSAO}.csv"))
    assert len(df) == len(particao), f"{area}: dataset e partição divergem"
    if "run_idx" not in df.columns:
        df["run_idx"] = np.arange(len(df))
    entradas = [c for c in df.columns if c.startswith(PREFIXOS_FEATURES)]
    alvos = [c for c in df.columns if c in LIMITE_DE]            # só termopares
    ignoradas = [c for c in df.columns if c not in entradas + alvos + ["run_idx"]]
    assert alvos, f"{area}: nenhum termopar reconhecido"
    inesperadas = [c for c in ignoradas if not c.startswith(EXCLUIDAS)]
    assert not inesperadas, (
        f"{area}: colunas que não são entrada nem alvo: {inesperadas}. "
        "Declare o prefixo em PREFIXOS_FEATURES ou, se a exclusão for deliberada, em EXCLUIDAS.")

    longo = (df[df["run_idx"].isin(IDX_TREINO)]
             .melt(id_vars=entradas + ["run_idx"], value_vars=alvos,
                   var_name="componente", value_name="temperatura")
             .dropna(subset=["temperatura"])          # máscara: alvo ausente não entra
             .reset_index(drop=True))
    grupos = longo["run_idx"].map(DOBRA).astype(int)

    chave = longo[entradas].astype(str).agg("|".join, axis=1)
    assert (grupos.groupby(chave).nunique() == 1).all(), f"{area}: configuração em mais de uma dobra"

    return longo[entradas + ["componente"]], longo["temperatura"].to_numpy(), grupos.to_numpy(), longo

## 3. Patamar e escala, pipelines e espaços de busca

In [ ]:
class EstatisticasComponente(BaseEstimator, TransformerMixin):
    """Substitui a identidade do componente por patamar e escala, ajustados apenas no treino da dobra."""

    def fit(self, X, y):
        y = pd.Series(np.asarray(y, dtype=float), index=X.index)
        g = y.groupby(X["componente"])
        # Mesma regra do caderno de modelagem: escala 1 quando o desvio nao pode ser calculado
        # (um unico valor no treino) ou quando e zero. Antes era 0 no primeiro caso.
        self.patamar_, self.escala_ = g.mean(), g.std().fillna(1.0).replace(0, 1.0)
        self.patamar_global_, self.escala_global_ = float(y.mean()), float(y.std())
        return self

    def transform(self, X):
        comp = X["componente"]
        out = X.drop(columns="componente").copy()
        out["patamar"] = comp.map(self.patamar_).fillna(self.patamar_global_).to_numpy()
        out["escala"] = comp.map(self.escala_).fillna(self.escala_global_).to_numpy()
        return out


MODELOS = {
    "ridge": lambda: Ridge(),
    "random_forest": lambda: RandomForestRegressor(random_state=SEMENTE, n_jobs=1),
    "extra_trees": lambda: ExtraTreesRegressor(random_state=SEMENTE, n_jobs=1),
    "gradient_boosting": lambda: GradientBoostingRegressor(random_state=SEMENTE),
}


def cria_pipeline(algoritmo):
    passos = [("estatisticas", EstatisticasComponente())]
    if algoritmo == "ridge":
        passos.append(("escala", StandardScaler()))   # só o modelo linear depende de escala
    passos.append(("modelo", MODELOS[algoritmo]()))
    return Pipeline(passos)


# Configurações da Sprint 3, avaliadas no mesmo protocolo como referência
BASE = {
    "ridge": {"modelo__alpha": 1.0},
    "random_forest": {"modelo__n_estimators": 300},
    "extra_trees": {"modelo__n_estimators": 300},
    "gradient_boosting": {"modelo__n_estimators": 300, "modelo__learning_rate": 0.05,
                          "modelo__max_depth": 2, "modelo__min_samples_leaf": 2,
                          "modelo__subsample": 0.8},
}

_arvores = {
    "modelo__n_estimators": randint(100, 601),
    "modelo__max_depth": [None, 4, 6, 8, 12, 16],
    "modelo__min_samples_leaf": randint(1, 11),
    "modelo__min_samples_split": randint(2, 11),
    "modelo__max_features": [1.0, 0.8, 0.6, 0.5, 0.33],
}
ESPACOS = {
    "ridge": {"modelo__alpha": loguniform(1e-3, 1e3)},
    "random_forest": {**_arvores, "modelo__max_samples": [None, 0.6, 0.8]},
    "extra_trees": dict(_arvores),
    "gradient_boosting": {
        "modelo__n_estimators": randint(100, 801),
        "modelo__learning_rate": loguniform(0.01, 0.2),
        "modelo__max_depth": randint(2, 7),
        "modelo__min_samples_leaf": randint(1, 16),
        "modelo__subsample": uniform(0.5, 0.5),        # intervalo [0.5, 1.0]
        "modelo__max_features": [None, 0.8, 0.6],
        "modelo__loss": ["squared_error", "huber", "absolute_error"],
    },
}

## 4. KPIs e avaliação de uma configuração

In [ ]:
def kpis(longo, previsto):
    """Métricas sobre as previsões acumuladas das 17 dobras."""
    real = longo["temperatura"].to_numpy()
    erro = previsto - real
    r2 = [r2_score(real[m], previsto[m])
          for m in (longo["componente"] == c for c in longo["componente"].unique())
          if np.ptp(real[m]) > 0]

    lim = longo["componente"].map(LIMITE_DE).to_numpy(dtype=float)
    aplica = ~np.isnan(lim) & ~longo["componente"].isin(SO_MONITORAR).to_numpy()
    excesso = aplica & (real >= np.nan_to_num(lim, nan=np.inf))

    res = {"n_previsoes": len(real),
           "MAE": mean_absolute_error(real, previsto),
           "RMSE": float(np.sqrt(mean_squared_error(real, previsto))),
           "MAPE_%": float(100 * np.mean(np.abs(erro) / real)),
           "dentro_margem_%": float(100 * np.mean(np.abs(erro) / real <= MARGEM_KPI / 100)),
           "vies": float(erro.mean()),
            "R2_area": float(r2_score(real, previsto)),
           "R2_medio_comp": float(np.mean(r2)) if r2 else np.nan,
           "excessos": int(excesso.sum())}
    for m in MARGENS_SEGURANCA:
        res[f"FA_m{m:g}"] = int((excesso & (previsto + m < lim)).sum())
    return res


def avalia(algoritmo, params, X, y, grupos, longo):
    pipe = cria_pipeline(algoritmo).set_params(**params)
    t0 = time.perf_counter()
    prev = cross_val_predict(pipe, X, y, groups=grupos, cv=LeaveOneGroupOut(), n_jobs=N_JOBS)
    return {**kpis(longo, prev), "tempo_s": round(time.perf_counter() - t0, 1)}

## 5. Busca aleatória por área e algoritmo

A busca ordena os candidatos pelo MAE médio entre dobras (critério do `RandomizedSearchCV`). O melhor candidato é o de posição 1 nesse critério; os `TOP_K` primeiros e a configuração da Sprint 3 são então reavaliados com as previsões acumuladas, que dão o MAE comparável ao registro da Sprint 3 e os falsos aprovados.

In [ ]:
def _json(o):
    return o.item() if hasattr(o, "item") else str(o)


def busca(area, algoritmo, X, y, grupos, longo):
    rs = RandomizedSearchCV(cria_pipeline(algoritmo), ESPACOS[algoritmo],
                            n_iter=N_ITER[algoritmo], scoring="neg_mean_absolute_error",
                            cv=LeaveOneGroupOut(), n_jobs=N_JOBS, random_state=SEMENTE,
                            refit=False, return_train_score=True, error_score="raise")
    t0 = time.perf_counter()
    rs.fit(X, y, groups=grupos)
    t_busca = time.perf_counter() - t0

    cv = pd.DataFrame(rs.cv_results_)
    cv["MAE_val_dobras"] = -cv["mean_test_score"]
    cv["MAE_treino"] = -cv["mean_train_score"]
    cv = cv.sort_values("rank_test_score").reset_index(drop=True)
    cv.drop(columns=[c for c in cv.columns if c.startswith("split")]).to_csv(
        os.path.join(PASTA_SAIDA, f"cv_{area}_{algoritmo}.csv"), index=False)

    candidatos = [("sprint3", BASE[algoritmo])] + [
        (f"rs_top{i + 1}", cv.loc[i, "params"]) for i in range(min(TOP_K, len(cv)))]
    linhas = []
    for rotulo, params in candidatos:
        linhas.append({"area": area, "algoritmo": algoritmo, "candidato": rotulo,
                       **avalia(algoritmo, params, X, y, grupos, longo),
                       "params": json.dumps({k.replace("modelo__", ""): v
                                             for k, v in params.items()}, default=_json)})
    tab = pd.DataFrame(linhas)
    tab["tempo_busca_s"] = round(t_busca, 1)
    tab.to_csv(os.path.join(PASTA_SAIDA, f"kpis_{area}_{algoritmo}.csv"), index=False)
    return tab

## 6. Execução

Com os valores padrão de `N_ITER`, são 140 candidatos por área, cada um ajustado 17 vezes. Para uma rodada de conferência, reduza `N_ITER` ou restrinja `AREAS` e `ALGORITMOS` na célula de configuração. Com `RETOMAR = True`, uma execução interrompida continua do ponto em que parou.

In [ ]:
tabelas, LINHAS_AREA = [], {}
for area in AREAS:
    X, y, grupos, longo = carrega_longo(area)
    LINHAS_AREA[area] = len(longo)
    print(f"\n{area}: {len(longo)} linhas | {longo['componente'].nunique()} componentes "
          f"| {len(np.unique(grupos))} dobras")
    for alg in ALGORITMOS:
        arq = os.path.join(PASTA_SAIDA, f"kpis_{area}_{alg}.csv")
        if RETOMAR and os.path.exists(arq):
            tab = pd.read_csv(arq)
            print(f"  {alg:<18} retomado")
        else:
            tab = busca(area, alg, X, y, grupos, longo)
            b, m = tab.iloc[0], tab.iloc[1]
            print(f"  {alg:<18} Sprint 3 {b.MAE:6.2f} °C | RS {m.MAE:6.2f} °C "
                  f"| FA@8 {b.FA_m8:>3} -> {m.FA_m8:<3} | {m.tempo_busca_s:.0f} s")
        tabelas.append(tab)

KPIS = pd.concat(tabelas, ignore_index=True)
divergentes = KPIS[KPIS["n_previsoes"] != KPIS["area"].map(LINHAS_AREA)]
assert divergentes.empty, (
    "n_previsoes diferente do dataset da área (cache antigo em PASTA_SAIDA?):\n"
    + divergentes[["area", "algoritmo", "candidato", "n_previsoes"]].to_string(index=False))
print("\nLeituras por área:", LINHAS_AREA, "| total:", sum(LINHAS_AREA.values()))
KPIS.to_csv(os.path.join(PASTA_SAIDA, "kpis_consolidado.csv"), index=False)

## 7. Consolidação

In [ ]:
# Melhor da busca (top 1) contra a configuração da Sprint 3, por área e algoritmo
base = KPIS[KPIS["candidato"] == "sprint3"].set_index(["area", "algoritmo"])
melhor = KPIS[KPIS["candidato"] == "rs_top1"].set_index(["area", "algoritmo"])

comparacao = pd.DataFrame({
    "R2_area_sprint3": base["R2_area"], "R2_area_rs": melhor["R2_area"],
    "MAE_sprint3": base["MAE"], "MAE_rs": melhor["MAE"],
    "ganho_%": 100 * (base["MAE"] - melhor["MAE"]) / base["MAE"],
    "dentro_margem_rs_%": melhor["dentro_margem_%"],
    "R2_rs": melhor["R2_medio_comp"],
    "FA_m8_sprint3": base["FA_m8"], "FA_m8_rs": melhor["FA_m8"],
    "excessos": melhor["excessos"],
}).round(2)
print(comparacao.to_string())

In [ ]:
# MAE do melhor candidato: áreas nas linhas, algoritmos nas colunas
print(melhor["MAE"].unstack("algoritmo").reindex(AREAS)[ALGORITMOS].round(2).to_string())

In [ ]:
# Agregado das sete áreas por algoritmo, ponderado pelo número de previsões
def agrega(t):
    n = t["n_previsoes"]
    return pd.Series({"MAE": (t["MAE"] * n).sum() / n.sum(),
                      "dentro_margem_%": (t["dentro_margem_%"] * n).sum() / n.sum(),
                      "excessos": t["excessos"].sum(),
                      **{f"FA_m{m:g}": t[f"FA_m{m:g}"].sum() for m in MARGENS_SEGURANCA}})

agregado = (KPIS[KPIS["candidato"].isin(["sprint3", "rs_top1"])]
            .groupby(["algoritmo", "candidato"]).apply(agrega).round(2))
print(agregado.to_string())

In [ ]:
# Troca entre precisão e segurança nos candidatos do topo
print(KPIS[["area", "algoritmo", "candidato", "MAE", "dentro_margem_%", "FA_m0", "FA_m8"]]
      .round(2).to_string(index=False))

## 8. Leitura dos resultados

Quando `supera_sprint3` for falso, a busca não encontrou configuração melhor que a da Sprint 3 naquela área, e a entrega deve manter a configuração anterior. O MAE do candidato vencedor é otimista, porque a mesma validação que escolheu o candidato também o avalia. O ganho sobre a configuração da Sprint 3 só deve ser tratado como real quando superar a diferença de 0,5 °C adotada como limiar de ruído do projeto, e a comparação com o grid search é válida por estar sujeita ao mesmo viés, desde que ambos usem este protocolo.

A execução atual, de 24 de setembro de 2026, é a primeira com as seis colunas do motivo do ensaio entre as entradas. A execução anterior as descartava, por um prefixo não declarado, e os seus números, citados no relatório de resultados desta busca, não devem mais ser usados. Com o motivo presente, o gradiente buscado tem o menor erro nas sete áreas, e em Main Vent Sys o Extra Trees, que antes aparecia com 3,77 graus, mede 3,89, logo atrás do gradiente, com 3,87. Somadas as sete áreas, o gradiente buscado mede 5,87 graus, com 31 aprovações indevidas na margem de oito graus. Esses valores têm o otimismo descrito acima, por serem o melhor de quarenta candidatos avaliado nas mesmas dobras em que foi escolhido; a formulação entregue, medida sem esse otimismo pela regra fixa, mede 5,86 graus com 24 aprovações indevidas, e o gradiente buscado, colocado dentro dela e medido pelo aninhamento estrito, piorou o erro.
